In [2]:
from dotenv import load_dotenv
import os
load_dotenv()
from groq import Groq
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [3]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [4]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=client,
    instructions=instructions,
)

In [5]:
assistant.rag("How do I run Olama locally?")

'To run Olama locally, there is no information in the provided context.'

In [40]:
messages = [
    {"role": "user", "content": 'I just discovered the course. Can I still join?'},
    ]

In [5]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [16]:
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"]
        }
    }
}

In [17]:
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=messages,
    tools=[search_tool],
    tool_choice="auto"
)

In [10]:
len(response.choices)

response.choices

[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='<brave_search>Is the course still accepting new members?</brave_search>', role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=None))]

In [18]:
import json

# First check the finish_reason and that tool_calls exists
choice = response.choices[0]

if choice.finish_reason == "tool_calls" and choice.message.tool_calls:
    tool_call = choice.message.tool_calls[0]
    args = json.loads(tool_call.function.arguments)
    print(args)

{'query': 'join course late'}


In [19]:
results = search(**args)

In [21]:
result_json = json.dumps(results, indent=2)

In [22]:
print(result_json) #this is what we're sending back to the llm

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "bd31146b0e",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "When will the course be offered next?",
    "answer": "Summer 2025."
  },
  {
    "id": "69d122f12e",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",
    "answer": "No, you can only get a certificate if you finish the course with a \"live\" cohort.\n\nWe don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review p

In [41]:
# Add the assistant's response with tool calls
messages.append({
    "role": "assistant",
    "content": None,
    "tool_calls": choice.message.tool_calls
})

# Add the tool result
messages.append({
    "role": "tool",
    "tool_call_id": choice.message.tool_calls[0].id,
    "content": result_json
})

In [42]:
messages

[{'role': 'user',
  'content': 'I just discovered the course. Can I still join?'},
 {'role': 'assistant',
  'content': None,
  'tool_calls': [ChatCompletionMessageToolCall(id='nnr3nbzgm', function=Function(arguments='{"query":"join course late"}', name='search'), type='function')]},
 {'role': 'tool',
  'tool_call_id': 'nnr3nbzgm',
  'content': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "bd31146b0e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "When will the course be offered next?",\n    "answer": "Summer 2025."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n

In [43]:
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=messages,
    tools=[search_tool]
)

In [45]:
response.choices[0].message.content.strip()

'Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [47]:
usage = response.usage
print(usage)

CompletionUsage(completion_tokens=33, prompt_tokens=813, total_tokens=846, completion_time=0.04367875, completion_tokens_details=None, prompt_time=0.082544413, prompt_tokens_details=None, queue_time=0.02090167, total_time=0.126223163)
